# Specimen 01 — Researcher → Writer Handoff

Goal: two agents, one hands its output to the other. Learn the shape of an agent-to-agent handoff before adding more roles.

In [1]:
import os
import json
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
MODEL = 'claude-opus-5'

INPUT_PRICE_PER_MTOK = 5.00
OUTPUT_PRICE_PER_MTOK = 25.00

def call_cost(usage):
    return (usage.input_tokens / 1_000_000 * INPUT_PRICE_PER_MTOK) + (usage.output_tokens / 1_000_000 * OUTPUT_PRICE_PER_MTOK)

def call_model(messages, tools=None, max_tokens=800, output_schema=None):
    kwargs = dict(model=MODEL, max_tokens=max_tokens, messages=messages, thinking={"type": "disabled"})
    if tools:
        kwargs['tools'] = tools
    if output_schema:
        kwargs['output_config'] = {"format": {"type": "json_schema", "schema": output_schema}}
    return client.messages.create(**kwargs)


`call_model` bakes in two things learned the hard way in Phase 3/4: `thinking` is disabled by default, since `claude-opus-5` defaults to extended thinking and can silently burn a small `max_tokens` budget before producing a visible tool call or text block; and `output_schema` makes structured JSON handoffs a one-liner, since Phase 3's chain-of-thought grading bug came from regex-parsing freeform prose instead of forcing a schema. Every agent-to-agent handoff in this phase should go through `output_schema`, not string parsing.

## 1. Define the researcher agent

Give it a fixed local knowledge source to search (reuse a Phase 4-style tool, or just a small dict) and a system prompt establishing its role.

In [2]:
local_knowledge_base = {
    "lithium_ion": "Lithium-ion batteries offer high energy density (250-300 Wh/kg) and mature manufacturing, but rely on constrained lithium/cobalt/nickel supply chains and degrade after roughly 500-1000 charge cycles.",
    "solid_state": "Solid-state batteries replace the liquid electrolyte with a solid one, promising higher energy density and much better safety since there's no flammable liquid, but as of 2026 they remain expensive and hard to manufacture at scale.",
    "sodium_ion": "Sodium-ion batteries use abundant, cheap sodium instead of lithium, trading lower energy density (around 150 Wh/kg) for much lower cost and notably better cold-weather performance.",
    "lfp": "Lithium iron phosphate (LFP) batteries sacrifice some energy density for a much longer cycle life (2000+ cycles), lower cost, and better thermal stability than standard lithium-ion.",
}

RESEARCHER_SYSTEM = (
    "You are a research agent. You are given a topic key and a fixed knowledge-base entry for it. "
    "Produce a structured research note based ONLY on that entry -- do not invent facts beyond what's provided."
)

print(f"Knowledge base ready: {list(local_knowledge_base.keys())}")

Knowledge base ready: ['lithium_ion', 'solid_state', 'sodium_ion', 'lfp']


## 2. Make it hand off a structured note, not prose

Use `call_model`'s `output_schema` to force a JSON shape (e.g. `{summary, key_facts: [...], open_questions: [...]}`). The writer agent should never have to regex-parse the researcher's prose — this is the direct lesson from Phase 3's chain-of-thought extraction bug.

In [3]:
RESEARCH_NOTE_SCHEMA = {
    "type": "object",
    "properties": {
        "summary": {"type": "string"},
        "key_facts": {"type": "array", "items": {"type": "string"}},
        "open_questions": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["summary", "key_facts", "open_questions"],
    "additionalProperties": False,
}

def run_researcher(topic_key):
    entry = local_knowledge_base.get(topic_key)
    if entry is None:
        raise KeyError(f"No knowledge base entry for '{topic_key}'")
    response = call_model(
        messages=[{"role": "user", "content": (
            f"{RESEARCHER_SYSTEM}\n\nTopic: {topic_key}\nKnowledge base entry: {entry}\n\n"
            "Produce a structured research note."
        )}],
        output_schema=RESEARCH_NOTE_SCHEMA,
    )
    text = ''.join(b.text for b in response.content if b.type == 'text')
    return json.loads(text), response.usage

note, usage = run_researcher("solid_state")
print(json.dumps(note, indent=2))

{
  "summary": "Solid-state batteries substitute the liquid electrolyte of conventional cells with a solid electrolyte. This design promises higher energy density and substantially improved safety, since eliminating the flammable liquid removes a key fire risk. However, as of 2026 the technology remains costly and difficult to manufacture at scale, which limits commercial deployment.",
  "key_facts": [
    "Solid-state batteries replace the liquid electrolyte with a solid electrolyte.",
    "They promise higher energy density than conventional liquid-electrolyte batteries.",
    "Safety is much better because there is no flammable liquid in the cell.",
    "As of 2026, solid-state batteries remain expensive.",
    "As of 2026, they are hard to manufacture at scale."
  ],
  "open_questions": [
    "How much higher is the energy density in quantitative terms compared with liquid-electrolyte lithium-ion cells?",
    "Which solid electrolyte materials (e.g., ceramic, polymer, sulfide) are 

## 3. Define the writer agent

Takes the researcher's structured note as input, drafts a short report from it — nothing else.

In [4]:
WRITER_SYSTEM = (
    "You are a writer agent. You are given a structured research note (summary, key_facts, "
    "open_questions) produced by a researcher agent. Draft a short report (3-5 sentences) based "
    "only on that note -- do not add outside knowledge."
)

def run_writer(research_note):
    response = call_model(
        messages=[{"role": "user", "content": (
            f"{WRITER_SYSTEM}\n\nResearch note (JSON):\n{json.dumps(research_note)}\n\n"
            "Write the report now."
        )}],
        max_tokens=400,
    )
    text = ''.join(b.text for b in response.content if b.type == 'text')
    return text, response.usage

report, usage = run_writer(note)
print(report)

**Report: Solid-State Batteries**

Solid-state batteries replace the liquid electrolyte used in conventional cells with a solid electrolyte, a change that promises higher energy density than existing liquid-electrolyte designs. The approach also offers substantially improved safety, since removing the flammable liquid eliminates a key fire risk within the cell. As of 2026, however, the technology remains expensive and difficult to manufacture at scale, and these constraints continue to limit commercial deployment.

Several questions remain unresolved by the current note. It does not quantify the energy-density advantage over conventional lithium-ion cells, identify which solid electrolyte materials (ceramic, polymer, or sulfide) are in play, or specify the manufacturing bottlenecks, cost figures, and likely timeline to cost parity. Also unaddressed are the first target applications and leading developers, and whether durability or safety concerns such as dendrite formation, interfacial

## 4. Chain them

Researcher runs on a fixed question, its structured output feeds directly into the writer's prompt, print the final report.

In [5]:
def researcher_writer_pipeline(topic_key):
    note, usage_r = run_researcher(topic_key)
    report, usage_w = run_writer(note)
    return report, note, usage_r, usage_w

report, note, usage_r, usage_w = researcher_writer_pipeline("solid_state")
print("RESEARCH NOTE:")
print(json.dumps(note, indent=2))
print("\nFINAL REPORT:")
print(report)

RESEARCH NOTE:
{
  "summary": "Solid-state batteries substitute a solid electrolyte for the conventional liquid electrolyte. The knowledge-base entry attributes two main benefits to this change: higher energy density and substantially improved safety, the latter because no flammable liquid is present. The offsetting drawback is commercial immaturity: as of 2026 the technology is still expensive and difficult to manufacture at scale, which frames it as promising but not yet mass-market ready.",
  "key_facts": [
    "Solid-state batteries replace the liquid electrolyte used in conventional batteries with a solid electrolyte.",
    "They promise higher energy density than liquid-electrolyte batteries.",
    "They promise much better safety, because there is no flammable liquid electrolyte.",
    "As of 2026, solid-state batteries remain expensive.",
    "As of 2026, they remain hard to manufacture at scale."
  ],
  "open_questions": [
    "How much higher is the energy density in quantita

## 5. Vary the question

Run the same pipeline on 2-3 different questions. Confirm the handoff format holds up every time — not just on the one question you designed it around.

In [6]:
for topic_key in ["lithium_ion", "sodium_ion", "lfp"]:
    report, note, _, _ = researcher_writer_pipeline(topic_key)
    print(f"=== {topic_key} ===")
    print(report)
    print()

=== lithium_ion ===
**Report: Lithium-Ion Battery Technology Status**

Lithium-ion batteries represent a mature energy storage technology, offering high energy density in the range of 250–300 Wh/kg and backed by established manufacturing capacity. Their principal limitations are twofold: a dependence on constrained supply chains for lithium, cobalt, and nickel, and cell degradation that emerges after roughly 500–1000 charge cycles.

Several aspects of this profile remain unresolved in the source material. It is not specified what makes the lithium, cobalt, and nickel supply chains constrained — whether geographic concentration, extraction capacity, geopolitics, or some combination — nor how degradation is defined at the 500–1000 cycle mark, such as by a percent capacity retention threshold. Likewise, the effect of different chemistry variants within the lithium-ion family on the stated energy density and cycle-life figures is not addressed, and the entry provides no information on cost

=== sodium_ion ===
# Sodium-Ion Batteries: A Cost-Driven Trade-Off

Sodium-ion batteries replace lithium with sodium as the working ion, drawing on a material that is both abundant and inexpensive by comparison. This substitution comes at the cost of energy density, which sits at roughly 150 Wh/kg — comparatively low relative to lithium-based alternatives. In exchange, the technology offers substantially lower cost and notably better performance in cold weather, making the design an explicit trade-off rather than an across-the-board improvement.

Several questions remain unresolved in the current note. Key among them are the specific cathode, anode, and electrolyte chemistries in use; how cycle and calendar life compare to lithium-ion; and the quantified cost advantage per kWh. Also outstanding are the precise temperature and capacity-retention figures behind the cold-weather claim, which applications (such as grid storage or entry-level EVs) best suit the ~150 Wh/kg range, the present

=== lfp ===
# Lithium Iron Phosphate (LFP): A Durability-Focused Battery Chemistry

Lithium iron phosphate (LFP) is a battery chemistry within the lithium-ion family that is best understood as a deliberate engineering trade-off: it accepts lower energy density than standard lithium-ion batteries in exchange for gains in durability, cost, and safety. In return for that reduced energy stored per unit mass or volume, LFP delivers a substantially longer cycle life — cited at 2000+ cycles — along with a lower cost and better thermal stability, which implies improved safety margins. These characteristics position LFP as the favorable choice in applications where longevity, affordability, and safety matter more than maximizing energy density.

Several important details remain unresolved in the current research. Notably, the actual energy density figures and the size of the gap versus NMC/NCA chemistries are not established, nor are the test conditions (depth of discharge, charge rate, tempera

## 6. Track cost across both agents

Sum `call_cost()` across every call in one pipeline run — continuing the habit from Phase 3/4, not bolting it on later.

In [7]:
total_cost = 0.0
for topic_key in local_knowledge_base:
    report, note, usage_r, usage_w = researcher_writer_pipeline(topic_key)
    total_cost += call_cost(usage_r) + call_cost(usage_w)

print(f"Total cost across all {len(local_knowledge_base)} researcher->writer pipeline runs: ${total_cost:.6f}")

Total cost across all 4 researcher->writer pipeline runs: $0.106690
